# Philippine National Multi-Disease Vaccination Campaign
## 01 — Dataset Column Definitions & Schema
**Synthetic Dataset | 2024–2028 Longitudinal | EIF Cohort 10 — Eskwelabs**

---
> This notebook defines the complete schema for all tables generated by `03_dataset_generator.ipynb`.  
> Each column includes: data type, constraints, generation method, sample values, and the model engine that produces it.  
> Use this as a data dictionary for both the generator and the EDA notebook.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# Display settings
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 120)

print('✅ Schema notebook ready.')
print('   Tables defined in this notebook:')
tables = [
    ('dim_sites',                'Dimension', 'Observable', '~150 rows'),
    ('dim_people',               'Dimension', 'Observable', '~5,000 rows'),
    ('fact_appointments',        'Fact',      'Observable', '~14,000 rows'),
    ('fact_vaccination_events',  'Fact',      'Observable', '~11,500 rows'),
    ('fact_inventory_shipments', 'Fact',      'Observable', '~1,200 rows'),
    ('gold_coverage',            'Gold',      'Derived',    '~2,040 rows'),
    ('gold_efficacy_waning',     'Gold',      'Derived',    '~50,000 rows'),
    ('gold_cold_chain_risk',     'Gold',      'Derived',    '~1,200 rows'),
]
for t, ttype, layer, size in tables:
    print(f'   {t:<30} {ttype:<12} {layer:<12} {size}')

---
## Table 1: `dim_sites` — Vaccination Facility Registry

**Layer:** Observable (Silver)  
**Grain:** One row per vaccination facility  
**Source Engine:** Generated once at simulation start; static across campaign years  
**Expected Rows:** ~150

In [ ]:
dim_sites_schema = pd.DataFrame([
    ('site_id',              'STRING',   'PRIMARY KEY',       'UUID v4',                          'PHV-001',            'Unique facility identifier'),
    ('site_name',            'STRING',   'NOT NULL',          'Faker PH + site_type suffix',      'Batangas RHU 3',     'Official facility name'),
    ('region',               'STRING',   'NOT NULL',          'Weighted sample: REGIONAL_POP_WEIGHTS', 'NCR',           '17 Philippine regions (PSA 2024)'),
    ('province',             'STRING',   'NOT NULL',          'Lookup from region → province map', 'Metro Manila',      '82 provinces'),
    ('city_municipality',    'STRING',   'NOT NULL',          'Lookup from province → LGU list',  'Quezon City',        'LGU name'),
    ('latitude',             'FLOAT',    '5.0 to 21.0',       'Uniform jitter around province centroid', 14.6760,       'WGS84 decimal degrees'),
    ('longitude',            'FLOAT',    '117.0 to 127.0',    'Uniform jitter around province centroid', 121.0437,      'WGS84 decimal degrees'),
    ('site_type',            'STRING',   'NOT NULL',          'Categorical: 4 types',             'RHU',                'Regional Hospital / RHU / Barangay Health Center / Mobile Outreach'),
    ('cold_chain_capacity',  'INTEGER',  '> 0',               'Normal(μ=80, σ=20), clipped [20,200]', 78,              'Max doses/day cold storage capacity'),
    ('operational_days_pw',  'INTEGER',  '1–7',               'Depends on site_type',             5,                   'Days per week facility operates'),
    ('rurality_index',       'FLOAT',    '0.0–1.0',           'Derived from region + site_type',  0.35,                '0=urban, 1=remote rural'),
    ('doh_accredited',       'BOOLEAN',  'NOT NULL',          'Bernoulli(0.92)',                   True,                'DOH accreditation status'),
    ('cold_chain_tier',      'STRING',   'NOT NULL',          'Derived from site_type',           'Tier 2',             'Tier 1 (regional) / Tier 2 (RHU) / Tier 3 (barangay) / Mobile'),
    ('established_year',     'INTEGER',  '1980–2023',         'Uniform integer',                  2005,                'Year facility was established'),
], columns=['column_name', 'dtype', 'constraints', 'generation_method', 'sample_value', 'description'])

print('dim_sites schema — 14 columns, ~150 rows')
display(dim_sites_schema)

---
## Table 2: `dim_people` — Beneficiary Registry

**Layer:** Observable (Silver)  
**Grain:** One row per registered beneficiary  
**Source Engine:** Population Distribution Engine (Engine A)  
**Expected Rows:** ~5,000

In [ ]:
dim_people_schema = pd.DataFrame([
    ('person_id',             'STRING',  'PRIMARY KEY',    'UUID v4',                                   'PH-00001',      'Unique beneficiary identifier'),
    ('full_name',             'STRING',  'NOT NULL',       'Faker PH Filipino name generator',          'Maria Santos',  'Beneficiary full name'),
    ('date_of_birth',         'DATE',    'NOT NULL',       'Derived from age_group + Uniform offset',   '1985-03-14',    'Date of birth (YYYY-MM-DD)'),
    ('age_group',             'STRING',  'NOT NULL',       'Categorical: α_g weights from Engine A',    '18-59',         '0-5 / 6-17 / 18-59 / 60+'),
    ('sex',                   'STRING',  'NOT NULL',       'Bernoulli(0.508) — PSA sex ratio',          'Female',        'Male / Female'),
    ('region',                'STRING',  'NOT NULL',       'Poisson-weighted: REGIONAL_POP_WEIGHTS',    'Region VII',    '17 PH regions'),
    ('province',              'STRING',  'NOT NULL',       'Lookup from region',                        'Cebu',          '82 provinces'),
    ('city_municipality',     'STRING',  'NOT NULL',       'Lookup from province',                      'Cebu City',     'LGU name'),
    ('barangay',              'STRING',  'NOT NULL',       'Faker PH barangay name',                    'Lahug',         'Barangay of residence'),
    ('socioeconomic_class',   'STRING',  'NOT NULL',       'Categorical: PSA 2022 SEC distribution',    'D',             'A / B / C / D / E'),
    ('priority_group',        'STRING',  'NOT NULL',       'Derived from age_group + SEC + occupation', 'General Pop',   'Healthcare Worker / Elderly / Indigent / General Population'),
    ('philhealth_member',     'BOOLEAN', 'NOT NULL',       'Bernoulli(f(SEC))',                         True,            'PhilHealth membership status'),
    ('distance_to_site_km',   'FLOAT',   '>= 0.0',         'LogNormal(μ=1.2, σ=0.8) × rurality_index', 3.4,            'Distance to assigned vaccination site (km)'),
    ('assigned_site_id',      'STRING',  'FOREIGN KEY',    'Nearest site by region + capacity weight',  'PHV-023',      'FK → dim_sites.site_id'),
    ('registration_date',     'DATE',    '>= 2024-01-01',  'Uniform over 2024-01 to 2024-06',           '2024-02-11',   'Date of registry enrollment'),
    ('has_comorbidity',       'BOOLEAN', 'NOT NULL',       'Bernoulli(f(age_group))',                    False,          'Presence of relevant comorbidity (diabetes, hypertension, etc.)'),
    ('prior_vaccination_hx',  'STRING',  'NULLABLE',       'Categorical: based on age + SEC',           'Partial',       'None / Partial / Complete — pre-campaign history'),
], columns=['column_name', 'dtype', 'constraints', 'generation_method', 'sample_value', 'description'])

print('dim_people schema — 17 columns, ~5,000 rows')
display(dim_people_schema)

---
## Table 3: `fact_appointments` — Scheduled Vaccination Slots

**Layer:** Observable (Silver)  
**Grain:** One row per scheduled dose appointment  
**Source Engine:** Behavioral Engine (Engine B) + Seasonal Forcing (Engine H)  
**Expected Rows:** ~14,000 (multiple appointments per person for multi-dose vaccines + annual flu shots over 5 years)

In [ ]:
fact_appointments_schema = pd.DataFrame([
    ('appt_id',               'STRING',  'PRIMARY KEY',    'UUID v4',                                       'APT-000001',    'Unique appointment identifier'),
    ('person_id',             'STRING',  'FOREIGN KEY',    'FK → dim_people.person_id',                     'PH-00001',      'Beneficiary'),
    ('site_id',               'STRING',  'FOREIGN KEY',    'FK → dim_sites.site_id',                        'PHV-023',       'Assigned facility'),
    ('vaccine_type',          'STRING',  'NOT NULL',       'Assigned per age_group + priority_group rules', 'MMR',           'COVID_Booster / Influenza / MMR / HPV / Hepatitis_B / Rabies_PEP'),
    ('dose_number',           'INTEGER', '>= 1',           'Sequential per person × vaccine_type',          2,               'Dose sequence number within the vaccine series'),
    ('scheduled_date',        'DATE',    '2024–2028',      'Engine H seasonal forcing + calendar mult.',    '2024-06-15',    'Planned administration date'),
    ('scheduled_year',        'INTEGER', '2024–2028',      'Derived from scheduled_date',                   2024,            'Year of scheduled appointment'),
    ('scheduled_month',       'INTEGER', '1–12',           'Derived from scheduled_date',                   6,               'Month of scheduled appointment'),
    ('season',                'STRING',  'NOT NULL',       'Derived from scheduled_date (PAGASA)',          'HABAGAT',       'AMIHAN / SUMMER / HABAGAT'),
    ('status',                'STRING',  'NOT NULL',       'Bernoulli(P(no-show)) from Engine B',           'Kept',          'Kept / No-show / Rescheduled / Cancelled'),
    ('noshow_probability',    'FLOAT',   '0.0–1.0',        'Logistic model output — Engine B',              0.14,            'Model-computed P(no-show) for this appointment'),
    ('is_catch_up',           'BOOLEAN', 'NOT NULL',       'True if dose_number > 1 and prior no-show',     False,           'Catch-up appointment flag'),
    ('campaign_phase',        'STRING',  'NOT NULL',       'Derived from scheduled_date + DOH schedule',   'Phase 2',       'Phase 1 / Phase 2 / Phase 3 / Maintenance'),
    ('appointment_source',    'STRING',  'NOT NULL',       'Categorical: registration channel',             'Walk-in',       'Pre-registered / Walk-in / Community outreach / Referral'),
    ('reminder_sent',         'BOOLEAN', 'NOT NULL',       'Bernoulli(0.65)',                                True,           'Whether SMS/call reminder was sent prior to appointment'),
    ('distance_km',           'FLOAT',   '>= 0.0',         'Inherited from dim_people.distance_to_site_km', 3.4,            'Distance beneficiary must travel to this site'),
], columns=['column_name', 'dtype', 'constraints', 'generation_method', 'sample_value', 'description'])

print('fact_appointments schema — 16 columns, ~14,000 rows')
display(fact_appointments_schema)

---
## Table 4: `fact_vaccination_events` — Actual Administrations

**Layer:** Observable (Silver)  
**Grain:** One row per successfully administered dose  
**Source Engine:** Immunological Engine (C) + Pharmacovigilance Engine (E)  
**Expected Rows:** ~11,500 (~82% of appointments — those with status = 'Kept')

In [ ]:
fact_vaccination_events_schema = pd.DataFrame([
    ('event_id',              'STRING',  'PRIMARY KEY',    'UUID v4',                                       'EVT-000001',   'Unique event identifier'),
    ('appt_id',               'STRING',  'FOREIGN KEY',    'FK → fact_appointments.appt_id',                'APT-000001',   'Source appointment'),
    ('person_id',             'STRING',  'FOREIGN KEY',    'FK → dim_people.person_id',                     'PH-00001',     'Beneficiary'),
    ('site_id',               'STRING',  'FOREIGN KEY',    'FK → dim_sites.site_id',                        'PHV-023',      'Administration facility'),
    ('vaccine_type',          'STRING',  'NOT NULL',       'Inherited from fact_appointments',               'MMR',          'Vaccine administered'),
    ('dose_number',           'INTEGER', '>= 1',           'Inherited from fact_appointments',               2,              'Dose sequence number'),
    ('administered_date',     'DATE',    '2024–2028',      'scheduled_date + Normal(0, 1.5) day jitter',    '2024-06-15',   'Actual administration date'),
    ('lot_number',            'STRING',  'NOT NULL',       'Alphanumeric code: vaccine_type + batch ID',    'MMR-2024-B47', 'Vaccine lot/batch number'),
    ('expiry_date',           'DATE',    '> administered_date', 'Uniform [+6mo, +24mo] from manufacture',  '2025-12-01',   'Lot expiry date'),
    ('administered_by',       'STRING',  'NOT NULL',       'Categorical: staff type distribution',          'Nurse',        'Doctor / Nurse / Midwife / Health Aide'),
    ('route',                 'STRING',  'NOT NULL',       'Fixed per vaccine_type',                        'IM',           'IM / SC / ID / PO — per WHO schedule'),
    ('anatomical_site',       'STRING',  'NOT NULL',       'Fixed per route + age_group',                   'Deltoid (L)',  'Injection/administration site'),
    ('potency_at_admin',      'FLOAT',   '0.0–1.0',        'Arrhenius model × cold chain breach flag',      0.94,           'Estimated vaccine potency at time of administration'),
    ('efficacy_at_admin',     'FLOAT',   '0.0–1.0',        'E_peak × potency_at_admin × dose_prime_factor', 0.89,          'Estimated individual protection conferred'),
    ('days_since_prior_dose', 'INTEGER', 'NULLABLE',       'Null for dose 1; date diff for subsequent',     28,             'Days elapsed since previous dose in series'),
    ('series_complete',       'BOOLEAN', 'NOT NULL',       'True if dose_number == VACCINE_CONFIGS[doses]',  False,         'Whether this dose completes the recommended series'),
    ('next_dose_due_date',    'DATE',    'NULLABLE',       'administered_date + interval_days; Null if series_complete', '2024-07-13', 'Scheduled return for next dose'),
    ('aefi_reported',         'BOOLEAN', 'NOT NULL',       'Poisson(μ_v/1000) > 0',                         True,           'Whether any AEFI was reported'),
    ('aefi_severity',         'STRING',  'NULLABLE',       'Null if not aefi_reported; else Multinomial(π)', 'Mild',        'None / Mild / Moderate / Severe'),
    ('aefi_type',             'STRING',  'NULLABLE',       'Categorical per vaccine_type AEFI profile',     'Injection site pain', 'Specific AEFI description'),
    ('aefi_resolved',         'BOOLEAN', 'NULLABLE',       'Bernoulli(f(severity))',                         True,          'Whether AEFI resolved without sequelae'),
    ('cold_chain_ok',         'BOOLEAN', 'NOT NULL',       'Inherited from linked shipment_id',              True,          'Whether cold chain was intact for this dose'),
    ('linked_shipment_id',    'STRING',  'FOREIGN KEY',    'FK → fact_inventory_shipments.shipment_id',     'SHP-0042',     'Inventory batch used'),
], columns=['column_name', 'dtype', 'constraints', 'generation_method', 'sample_value', 'description'])

print('fact_vaccination_events schema — 23 columns, ~11,500 rows')
display(fact_vaccination_events_schema)

---
## Table 5: `fact_inventory_shipments` — Cold Chain Supply Movements

**Layer:** Observable (Silver)  
**Grain:** One row per shipment delivery to a facility  
**Source Engine:** Cold Chain Engine (D) + Logistics Engine (G)  
**Expected Rows:** ~1,200

In [ ]:
fact_inventory_schema = pd.DataFrame([
    ('shipment_id',           'STRING',  'PRIMARY KEY',    'UUID v4',                                       'SHP-0001',      'Unique shipment identifier'),
    ('site_id',               'STRING',  'FOREIGN KEY',    'FK → dim_sites.site_id',                        'PHV-023',       'Receiving facility'),
    ('vaccine_type',          'STRING',  'NOT NULL',       'One of 6 vaccine types',                        'COVID_Booster',  'Vaccine type in shipment'),
    ('source_hub',            'STRING',  'NOT NULL',       'Derived from region → DOH regional hub',        'DOH-NCR Hub',   'Regional DOH distribution hub'),
    ('shipment_date',         'DATE',    '2024–2028',      'Scheduled per replenishment model Engine G',    '2024-03-01',    'Date shipment dispatched from hub'),
    ('received_date',         'DATE',    '>= shipment_date','shipment_date + NegBinom(r=2, p=0.4) days',   '2024-03-03',    'Date shipment received at facility'),
    ('transit_days',          'INTEGER', '>= 0',           'received_date − shipment_date',                 2,               'Days in transit'),
    ('doses_ordered',         'INTEGER', '> 0',            'NegBinom(r=5, p=0.3) per site capacity',       500,             'Doses requested in order'),
    ('doses_received',        'INTEGER', '>= 0',           'doses_ordered × Uniform(0.80, 1.05)',           490,             'Actual doses received'),
    ('fulfillment_rate',      'FLOAT',   '0.0–1.05',       'doses_received / doses_ordered',                0.98,            'Order fulfillment ratio'),
    ('manufacture_date',      'DATE',    '< shipment_date','shipment_date − Uniform(30, 180) days',        '2023-12-01',    'Vaccine manufacture date'),
    ('expiry_date',           'DATE',    '> received_date','manufacture_date + vaccine shelf life',         '2025-06-01',    'Lot expiry date'),
    ('cold_chain_breach',     'BOOLEAN', 'NOT NULL',       'Bernoulli(ρ_site_type) — Engine D',             False,           'Whether temperature excursion occurred in transit'),
    ('breach_type',           'STRING',  'NULLABLE',       'Null if no breach; else categorical',           'Minor',         'None / Minor (2–8°C excursion) / Major (>8°C excursion)'),
    ('excursion_temp_c',      'FLOAT',   'NULLABLE',       'Null if no breach; else Normal(μ, σ) by type', 9.8,             'Peak temperature during excursion (°C)'),
    ('excursion_duration_hr', 'FLOAT',   'NULLABLE',       'Null if no breach; else Exponential(λ)',        4.5,             'Duration of temperature excursion (hours)'),
    ('potency_retained',      'FLOAT',   '0.0–1.0',        'Arrhenius model: e^(−k(T)×Δt)',                 0.87,            'Estimated vaccine potency after cold chain event'),
    ('wastage_rate',          'FLOAT',   '0.0–1.0',        'Beta(α_w, β_w) by breach severity — Engine D', 0.06,            'Proportion of received doses wasted'),
    ('doses_wasted',          'INTEGER', '>= 0',           'floor(doses_received × wastage_rate)',          29,              'Absolute doses wasted'),
    ('doses_available',       'INTEGER', '>= 0',           'doses_received − doses_wasted',                 461,             'Net usable doses from this shipment'),
    ('stock_on_hand_before',  'INTEGER', '>= 0',           'Running total from Engine G',                   120,             'Stock at facility before this shipment'),
    ('stock_on_hand_after',   'INTEGER', '>= 0',           'stock_on_hand_before + doses_available',        581,             'Stock after receiving this shipment'),
    ('stockout_flag',         'BOOLEAN', 'NOT NULL',       'True if stock_on_hand_after < safety_stock',    False,           'Stockout risk flag'),
    ('vvm_status',            'STRING',  'NOT NULL',       'Derived from potency_retained threshold',       'Stage 1',       'Vaccine Vial Monitor: Stage 1/2/3/4 (WHO VVM standard)'),
], columns=['column_name', 'dtype', 'constraints', 'generation_method', 'sample_value', 'description'])

print('fact_inventory_shipments schema — 24 columns, ~1,200 rows')
display(fact_inventory_schema)

---
## Gold Layer Tables — Derived Analytical Outputs

Gold tables are **not generated directly** — they are computed from the Silver tables by the generator's Master Factory cell.  
They represent the analytical outputs that power dashboards, EDA, and ML models.

In [ ]:
# --- gold_coverage ---
gold_coverage_schema = pd.DataFrame([
    ('coverage_id',          'STRING',  'PRIMARY KEY',  'UUID v4',                                        'COV-0001',   'Unique record identifier'),
    ('region',               'STRING',  'NOT NULL',     'FK → dim_sites.region',                          'NCR',        'PH region'),
    ('vaccine_type',         'STRING',  'NOT NULL',     'One of 6 vaccine types',                         'MMR',        'Vaccine type'),
    ('period_year',          'INTEGER', '2024–2028',    'Annual aggregation',                              2025,         'Reporting year'),
    ('period_quarter',       'INTEGER', '1–4',          'Quarterly aggregation',                           2,            'Reporting quarter'),
    ('target_population',    'INTEGER', '> 0',          'From dim_people filtered by age_targets',         18420,        'Eligible beneficiaries in region'),
    ('doses_administered',   'INTEGER', '>= 0',         'Count from fact_vaccination_events',              16540,        'Total doses given in period'),
    ('series_completed',     'INTEGER', '>= 0',         'Count where series_complete = True',              14200,        'Beneficiaries who completed full series'),
    ('coverage_rate',        'FLOAT',   '0.0–1.0',      'series_completed / target_population',           0.771,        'Vaccination coverage rate'),
    ('effective_coverage',   'FLOAT',   '0.0–1.0',      'Σ E_i(t) / target_population — Engine F',        0.682,        'Coverage adjusted for waning efficacy'),
    ('p_HIT',                'FLOAT',   '0.0–1.0',      'Fixed from VACCINE_CONFIGS',                     0.930,        'Herd immunity threshold for this vaccine'),
    ('herd_immunity_gap',    'FLOAT',   'Any',          'p_HIT − effective_coverage',                     0.248,        'Gap to herd immunity (negative = achieved)'),
    ('R_eff',                'FLOAT',   '>= 0.0',       'R0 × (1 − effective_coverage) — Engine F',       4.08,         'Effective reproduction number'),
    ('no_show_rate',         'FLOAT',   '0.0–1.0',      'no_shows / total_appointments in period',        0.183,        'Appointment no-show rate for period'),
    ('wastage_rate_avg',     'FLOAT',   '0.0–1.0',      'Avg wastage_rate from shipments in period',      0.061,        'Average vaccine wastage rate'),
], columns=['column_name', 'dtype', 'constraints', 'generation_method', 'sample_value', 'description'])

print('gold_coverage schema — 15 columns, ~2,040 rows (17 regions × 6 vaccines × 5 years × 4 quarters)')
display(gold_coverage_schema)

In [ ]:
# --- gold_efficacy_waning ---
gold_waning_schema = pd.DataFrame([
    ('waning_id',              'STRING', 'PRIMARY KEY',  'UUID v4',                                     'WAN-00001',  'Unique record'),
    ('person_id',              'STRING', 'FOREIGN KEY',  'FK → dim_people.person_id',                   'PH-00001',   'Beneficiary'),
    ('vaccine_type',           'STRING', 'NOT NULL',     'From vaccination event',                      'COVID_Booster', 'Vaccine'),
    ('final_dose_date',        'DATE',   'NOT NULL',     'Date of last dose in completed series',       '2024-03-20', 'Date series completed'),
    ('snapshot_date',          'DATE',   'NOT NULL',     'Monthly snapshots: final_dose_date + 30×n',  '2024-09-20', 'Date of efficacy snapshot'),
    ('days_since_final_dose',  'INTEGER','>=0',          'snapshot_date − final_dose_date',             184,          'Days elapsed since series completion'),
    ('estimated_protection',   'FLOAT',  '0.0–1.0',      'Bi-exponential: E_peak×[φ×e^(−λ_f×t)+(1−φ)×e^(−λ_s×t)]', 0.71, 'Current estimated protection'),
    ('booster_recommended',    'BOOLEAN','NOT NULL',     'True if estimated_protection < booster_trigger', False,     'Whether booster is now recommended'),
    ('protection_tier',        'STRING', 'NOT NULL',     'Derived from estimated_protection value',    'Moderate',   'High (>0.80) / Moderate (0.50–0.80) / Low (<0.50)'),
], columns=['column_name', 'dtype', 'constraints', 'generation_method', 'sample_value', 'description'])

print('gold_efficacy_waning schema — 9 columns, ~50,000 rows (monthly snapshots per vaccinated person × vaccine)')
display(gold_waning_schema)

# --- gold_cold_chain_risk ---
gold_cc_schema = pd.DataFrame([
    ('risk_id',              'STRING', 'PRIMARY KEY',  'UUID v4',                                       'CCR-0001',   'Unique record'),
    ('site_id',              'STRING', 'FOREIGN KEY',  'FK → dim_sites.site_id',                        'PHV-023',    'Facility'),
    ('shipment_id',          'STRING', 'FOREIGN KEY',  'FK → fact_inventory_shipments.shipment_id',     'SHP-0042',   'Shipment'),
    ('vaccine_type',         'STRING', 'NOT NULL',     'From shipment',                                  'Influenza',  'Vaccine type'),
    ('potency_retained',     'FLOAT',  '0.0–1.0',      'Inherited from fact_inventory_shipments',        0.87,         'Post-breach potency estimate'),
    ('doses_affected',       'INTEGER','>= 0',         'doses_received × (1 − potency_retained)',        64,           'Estimated sub-potent doses'),
    ('financial_loss_php',   'FLOAT',  '>= 0.0',       'doses_wasted × unit_cost_php[vaccine_type]',    12800.0,      'Estimated value of wasted doses (PHP)'),
    ('risk_tier',            'STRING', 'NOT NULL',     'Derived: Low/Medium/High/Critical',              'Medium',     'Cold chain risk classification'),
    ('action_required',      'STRING', 'NOT NULL',     'Rule-based from risk_tier + vvm_status',         'Quarantine', 'None / Monitor / Quarantine / Discard'),
], columns=['column_name', 'dtype', 'constraints', 'generation_method', 'sample_value', 'description'])

print('gold_cold_chain_risk schema — 9 columns, ~1,200 rows (one per shipment)')
display(gold_cc_schema)

---
## Schema Summary & Entity Relationships

The diagram below follows **Crow's Foot notation** (IEEE/Chen ERD standard), rendered via Mermaid.js.

```mermaid
erDiagram
    dim_sites {
        string site_id PK
        string site_name
        string region
        string province
        string city_municipality
        float latitude
        float longitude
        string site_type
        int cold_chain_capacity
        int operational_days_pw
        float rurality_index
        boolean doh_accredited
        string cold_chain_tier
        int established_year
    }

    dim_people {
        string person_id PK
        string full_name
        date date_of_birth
        string age_group
        string sex
        string region
        string province
        string socioeconomic_class
        string priority_group
        boolean philhealth_member
        float distance_to_site_km
        string assigned_site_id FK
        date registration_date
        boolean has_comorbidity
        string prior_vaccination_hx
    }

    fact_appointments {
        string appt_id PK
        string person_id FK
        string site_id FK
        string vaccine_type
        int dose_number
        date scheduled_date
        string season
        string status
        float noshow_probability
        boolean is_catch_up
        string campaign_phase
        string appointment_source
        boolean reminder_sent
        float distance_km
    }

    fact_vaccination_events {
        string event_id PK
        string appt_id FK
        string person_id FK
        string site_id FK
        string vaccine_type
        int dose_number
        date administered_date
        string lot_number
        date expiry_date
        string administered_by
        string route
        float potency_at_admin
        float efficacy_at_admin
        boolean series_complete
        date next_dose_due_date
        boolean aefi_reported
        string aefi_severity
        boolean cold_chain_ok
        string linked_shipment_id FK
    }

    fact_inventory_shipments {
        string shipment_id PK
        string site_id FK
        string vaccine_type
        string source_hub
        date shipment_date
        date received_date
        int doses_received
        boolean cold_chain_breach
        string breach_type
        float potency_retained
        float wastage_rate
        int doses_wasted
        int doses_available
        int stock_on_hand_after
        boolean stockout_flag
        string vvm_status
    }

    gold_coverage {
        string coverage_id PK
        string region
        string vaccine_type
        int period_year
        int period_quarter
        int target_population
        int series_completed
        float coverage_rate
        float effective_coverage
        float herd_immunity_gap
        float R_eff
        float no_show_rate
        float wastage_rate_avg
    }

    gold_efficacy_waning {
        string waning_id PK
        string person_id FK
        string vaccine_type
        date final_dose_date
        date snapshot_date
        int days_since_final_dose
        float estimated_protection
        boolean booster_recommended
        string protection_tier
    }

    gold_cold_chain_risk {
        string risk_id PK
        string site_id FK
        string shipment_id FK
        string vaccine_type
        float potency_retained
        int doses_affected
        float financial_loss_php
        string risk_tier
        string action_required
    }

    dim_sites ||--o{ dim_people : "assigned to"
    dim_sites ||--o{ fact_appointments : "hosts"
    dim_sites ||--o{ fact_vaccination_events : "administers at"
    dim_sites ||--o{ fact_inventory_shipments : "receives"
    dim_sites ||--o{ gold_cold_chain_risk : "assessed at"
    dim_people ||--o{ fact_appointments : "schedules"
    dim_people ||--o{ fact_vaccination_events : "receives"
    dim_people ||--o{ gold_efficacy_waning : "tracked in"
    fact_appointments ||--o| fact_vaccination_events : "fulfilled by"
    fact_inventory_shipments ||--o{ fact_vaccination_events : "supplies doses to"
    fact_inventory_shipments ||--|| gold_cold_chain_risk : "assessed as"
```

### Table Summary

| Table | Layer | Expected Rows | Columns | Primary Key |
|---|---|---|---|---|
| `dim_sites` | Silver | ~150 | 14 | `site_id` |
| `dim_people` | Silver | ~5,000 | 17 | `person_id` |
| `fact_appointments` | Silver | ~14,000 | 16 | `appt_id` |
| `fact_vaccination_events` | Silver | ~11,500 | 23 | `event_id` |
| `fact_inventory_shipments` | Silver | ~1,200 | 24 | `shipment_id` |
| `gold_coverage` | Gold | ~2,040 | 15 | `coverage_id` |
| `gold_efficacy_waning` | Gold | ~50,000 | 9 | `waning_id` |
| `gold_cold_chain_risk` | Gold | ~1,200 | 9 | `risk_id` |
| **TOTAL** | | **~85,090** | **127** | |

---
## References

Schema design and column constraints are grounded in the following standards and references. Format: IEEE.

[1] World Health Organization, "Data Quality Review: A Toolkit for Facility Data Quality Assessment," WHO, Geneva, Switzerland, Tech. Rep. WHO/HIS/HSI/2017.8, 2017. [Online]. Available: https://www.who.int/healthinfo/tools_data_analysis/dqr_modules/en/

[2] Philippine Department of Health, "Field Health Services Information System (FHSIS) Manual of Operations," DOH Philippines, Manila, Tech. Rep., 2021. [Online]. Available: https://doh.gov.ph/fhsis

[3] World Health Organization, "WHO Vaccine Vial Monitor (VVM): Guidance for Procurement," WHO PQT, Geneva, 2022. [Online]. Available: https://www.who.int/teams/regulation-prequalification/regulation-and-safety/vvm

[4] Philippine Statistics Authority, "Philippine Standard Geographic Code (PSGC)," PSA, Manila, 2024. [Online]. Available: https://psa.gov.ph/classification/psgc

[5] Kimball, R. and Ross, M., *The Data Warehouse Toolkit: The Definitive Guide to Dimensional Modeling*, 3rd ed. Indianapolis, IN, USA: Wiley, 2013.

[6] Databricks, "Medallion Architecture: Best Practices for Data Lakehouse Layers," Databricks Documentation, 2023. [Online]. Available: https://docs.databricks.com/lakehouse/medallion.html

> **Note:** All foreign key relationships and grain definitions follow Kimball dimensional modeling conventions [5]. The Medallion Architecture layering (Silver → Gold) follows Databricks lakehouse best practices [6], adapted for public health administrative data structures per DOH FHSIS [2].